In [ ]:
# Dependencies are managed via requirements.txt
# !pip install optuna -q
# !pip install --upgrade lightgbm -q
# !pip install category_encoders


In [ ]:
import os
import sys
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn import set_config
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import StratifiedKFold, cross_val_score

import lightgbm as lgb
import xgboost as xgb
import optuna
from category_encoders import TargetEncoder

PROJECT_PATH = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
if str(PROJECT_PATH) not in sys.path:
    sys.path.insert(0, str(PROJECT_PATH))

set_config(transform_output="pandas")


In [ ]:
X_train_full = pd.read_parquet(PROJECT_PATH / 'X_train_full.parquet')
y_train = pd.read_parquet(PROJECT_PATH / 'y_train.parquet')['TARGET']


In [ ]:
skf = StratifiedKFold(n_splits = 5, shuffle=True, random_state=42)

In [ ]:
pcb_cols = [ col for col in X_train_full if col.startswith("PREV_")]
bereua_cols = [ col for col in X_train_full if col.startswith("BUREAU_") and not col.startswith("BUREAU_BB_")]
custom_fill_na_cols = pcb_cols + bereua_cols

In [ ]:
from Production.pipeline import (
    fillna_col,
    to_category,
    fillna_category,
    car_age,
    fix_day_employed,
    CustomFeatureBuilder
)

num_zero_transform = FunctionTransformer(fillna_col)
cat_with_na = Pipeline([
    ("fill_na", FunctionTransformer(fillna_category)),
    ("to_category", FunctionTransformer(to_category))
])
fill_car_age = FunctionTransformer(car_age)
fix_de = FunctionTransformer(fix_day_employed)
custom_feature_step = CustomFeatureBuilder()


In [ ]:
cat_cols = set(X_train_full.select_dtypes(include = ['string', 'object', 'category']).columns)
cat_cols_list = list(cat_cols)

In [ ]:
te_cols = ['OCCUPATION_TYPE', 'ORGANIZATION_TYPE']
target_encoder_step = TargetEncoder(
    cols = te_cols,
    smoothing = 10.0,
    cv = skf.n_splits
)

In [ ]:
preprocessor = ColumnTransformer (
    transformers = [
        ("tar_encd_col", target_encoder_step, te_cols),
        ("cat_col", cat_with_na, cat_cols_list),
        ("custom_col", num_zero_transform, custom_fill_na_cols),
    ],
    remainder="passthrough",)

In [ ]:
prep_pipeline = Pipeline([
    ("custom_feature", custom_feature_step),
    ("de_fix", fix_de),
    ("prep", preprocessor)
])
X_train_prep = prep_pipeline.fit_transform(X_train_full, y_train)

In [ ]:
def objective_lgbm(trial):
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 20, 64),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 150),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 150, 500),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-2, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 10.0, log=True),
    }

    clf_lgbm = lgb.LGBMClassifier(
        **params,
        random_state=17,
        n_jobs=4,
        verbose=-1
    )

    score = cross_val_score(clf_lgbm, X_train_prep, y_train, cv=skf, scoring='roc_auc', n_jobs=2).mean()
    return score

In [ ]:
study_lgbm = optuna.create_study(direction="maximize")
study_lgbm.optimize(objective_lgbm, n_trials=30)

In [ ]:
def objective_xgb(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 150, 450),
        "min_child_weight": trial.suggest_int("min_child_weight", 10, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-2, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 10.0, log=True),
    }

    clf_xgb = xgb.XGBClassifier(
        **params,
        tree_method="hist",
        device="cuda",
        random_state=17,
        n_jobs=4,
        verbosity=0,
        enable_categorical=True,
        eval_metric="auc"
    )

    score = cross_val_score(clf_xgb, X_train_prep, y_train, cv=skf, scoring='roc_auc', n_jobs=2).mean()
    return score

In [ ]:
study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=30)

In [ ]:
best_params = {
    "params": {
        "lgbm": study_lgbm.best_params,
        "xgb": study_xgb.best_params
    },
    "scores": {
        "lgbm": float(study_lgbm.best_value),
        "xgb": float(study_xgb.best_value)
    }
}

out_params_path = PROJECT_PATH / 'best_params.json'
with open(out_params_path, 'w', encoding='utf-8') as f:
    json.dump(best_params, f, indent=4)
print(f'Best params saved to {out_params_path}')
